In [8]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from yolox.tracker.byte_tracker import BYTETracker
from collections import defaultdict

# ✅ Fix NumPy deprecation
if not hasattr(np, "float"):
    np.float = np.float64

# ✅ Paths
model_path = r"D:\Inam Project\inam\Project data\Results\weights\best.pt"
video_path = r"D:\Inam Project\inam\Project data\Simple Vdos-imgs\1.mp4"

# ✅ Load YOLOv11 model
model = YOLO(model_path)

# ✅ ByteTrack parameters
class TrackerArgs:
    track_thresh = 0.3
    track_buffer = 80
    match_thresh = 0.9
    aspect_ratio_thresh = 1.6
    min_box_area = 800
    mot20 = False

tracker = BYTETracker(TrackerArgs)

# ✅ Video setup
cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter("output_tracked.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

unique_ids = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ✅ YOLO detection
    results = model(frame, verbose=False)[0]

    detections = []
    for box in results.boxes:
        cls_id = int(box.cls)
        if cls_id != 0:
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        detections.append([x1, y1, x2, y2, conf, 1])  # last value = class or obj conf

    # ✅ Convert to PyTorch tensor for ByteTrack
    if len(detections) > 0:
        detections = torch.tensor(detections, dtype=torch.float32)
    else:
        detections = torch.zeros((0, 6), dtype=torch.float32)

    # ✅ Update tracker
    online_targets = tracker.update(detections, [h, w], (h, w))

    current_ids = set()
    for t in online_targets:
        tlwh = t.tlwh
        tid = t.track_id
        x1, y1, w_box, h_box = map(int, tlwh)
        x2, y2 = x1 + w_box, y1 + h_box

        current_ids.add(tid)
        unique_ids.add(tid)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f'ID {tid}', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.putText(frame, f'Current Pedestrians: {len(current_ids)}', (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.putText(frame, f'Total Unique: {len(unique_ids)}', (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    out.write(frame)
    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"✅ Finished. Total unique pedestrians detected: {len(unique_ids)}")


✅ Finished. Total unique pedestrians detected: 119
